In [ ]:
import sys
from pathlib import Path
from typing import cast

from omegaconf import DictConfig, OmegaConf

from utils.paths import get_root_path, get_source_path

# 경로 설정
project_root: Path = get_root_path()
source_root: Path = get_source_path()
sys.path.append(str(source_root))

# config 로드
config_path: Path = source_root / "conf" / "config.yaml"
cfg: DictConfig = cast(DictConfig, OmegaConf.load(config_path))
print(f"프로젝트 루트: {project_root}")
print(f"Source 루트: {source_root}")
print("Config 로드 완료")


In [ ]:
from typing import Dict, List, Union

from langchain_core.documents import Document
from langchain_text_splitters import MarkdownTextSplitter, RecursiveCharacterTextSplitter

CHUNKING_CONFIG: Dict[str, Dict[str, Union[int, bool]]] = {
    ".txt": {"chunk_size": 800, "chunk_overlap": 80},
    ".md": {"chunk_size": 1200, "chunk_overlap": 120, "use_markdown_splitter": True},
    ".org": {"chunk_size": 1000, "chunk_overlap": 100},
    ".rtf": {"chunk_size": 800, "chunk_overlap": 80},
    ".pdf": {"chunk_size": 1000, "chunk_overlap": 100},
    ".docx": {"chunk_size": 1000, "chunk_overlap": 150},
    ".doc": {"chunk_size": 1000, "chunk_overlap": 150},
    ".xlsx": {"chunk_size": 500, "chunk_overlap": 50},
    ".xls": {"chunk_size": 500, "chunk_overlap": 50},
    ".csv": {"chunk_size": 300, "chunk_overlap": 30},
    ".tsv": {"chunk_size": 300, "chunk_overlap": 30},
    ".pptx": {"chunk_size": 800, "chunk_overlap": 100},
    ".ppt": {"chunk_size": 800, "chunk_overlap": 100},
    ".hwp": {"chunk_size": 1000, "chunk_overlap": 100},
}

DEFAULT_CHUNK_SIZE: int = 1000
DEFAULT_CHUNK_OVERLAP: int = 200


def get_splitter(ext: str) -> Union[MarkdownTextSplitter, RecursiveCharacterTextSplitter]:
    """확장자에 따른 적절한 텍스트 스플리터 반환"""
    ext_config: Dict[str, Union[int, bool]] = CHUNKING_CONFIG.get(ext, {})
    chunk_size: int = ext_config.get("chunk_size", DEFAULT_CHUNK_SIZE)
    chunk_overlap: int = ext_config.get("chunk_overlap", DEFAULT_CHUNK_OVERLAP)
    use_markdown_splitter: bool = bool(ext_config.get("use_markdown_splitter", False))

    if use_markdown_splitter:
        return MarkdownTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


def chunk_documents(docs_by_file: List[List[Document]], ext: str) -> List[List[List[Document]]]:
    """파일별 Document들을 청킹"""
    chunked_by_file: List[List[List[Document]]] = []

    for file_docs in docs_by_file:
        splitter: Union[MarkdownTextSplitter, RecursiveCharacterTextSplitter] = get_splitter(ext)
        file_chunked_docs: List[List[Document]] = []

        for doc in file_docs:
            if isinstance(splitter, MarkdownTextSplitter):
                markdown_texts: List[str] = splitter.split_text(doc.page_content)
                markdown_chunks: List[Document] = [
                    Document(page_content=text) for text in markdown_texts
                ]
                file_chunked_docs.append(markdown_chunks)
            else:
                recursive_chunks: List[Document] = splitter.split_documents([doc])
                file_chunked_docs.append(recursive_chunks)

        chunked_by_file.append(file_chunked_docs)

    return chunked_by_file


In [ ]:
from typing import List

from langchain_core.documents import Document

from core.loader_router.loader import connect_loader

test_files_base: str = str(project_root / "notebooks" / "test-files" / "texts" / "md")
docs: List[List[Document]] = connect_loader(test_files_base, cfg=cfg, extensions=[".md"])[0]
docs_count: int = len(docs)
docs_count


In [ ]:
# 단일 파일 청킹 테스트
if docs:
    test_file_idx: int = 3
    single_file_docs: List[List[Document]] = [docs[test_file_idx]]
    example_ext: str = ".txt"

    print(f"선택된 파일: {test_file_idx + 1}번째")
    print(f"Document 수: {len(docs[test_file_idx])}")

    # 단일 파일 청킹
    chunked_result: List[List[List[Document]]] = chunk_documents(single_file_docs, example_ext)

    if chunked_result:
        file_result: List[List[Document]] = chunked_result[0]
        total_chunks: int = sum(len(doc_chunks) for doc_chunks in file_result)

        print(f"총 청크 수: {total_chunks}")

        for doc_idx, doc_chunks in enumerate(file_result):
            chunk_count: int = len(doc_chunks)
            print(f"Document {doc_idx + 1}: {chunk_count}개 청크")
else:
    print("로드된 문서가 없습니다.")


In [ ]:
# 모든 파일 청킹 테스트
if docs:
    example_ext: str = "."

    print(f"총 파일 수: {len(docs)}")

    # 모든 파일 청킹
    all_chunked_result: List[List[List[Document]]] = chunk_documents(docs, example_ext)

    if all_chunked_result:
        total_files: int = len(all_chunked_result)
        total_documents: int = 0
        total_chunks: int = 0

        for file_idx, file_result in enumerate(all_chunked_result):
            file_documents: int = len(file_result)
            file_chunks: int = sum(len(doc_chunks) for doc_chunks in file_result)

            total_documents += file_documents
            total_chunks += file_chunks

            print(f"파일 {file_idx + 1}: {file_documents}개 Document → {file_chunks}개 청크")

            for doc_idx, doc_chunks in enumerate(file_result):
                chunk_count: int = len(doc_chunks)
                print(f"  Document {doc_idx + 1}: {chunk_count}개 청크")

        print(
            f"총 파일 수: {total_files}, 총 Document 수: {total_documents}, 총 청크 수: {total_chunks}"
        )

        # 첫 번째 파일의 첫 번째 Document 청크 내용 출력
        if all_chunked_result[0] and all_chunked_result[0][0]:
            first_doc_chunks: List[Document] = all_chunked_result[0][0]
            for chunk_idx, chunk in enumerate(first_doc_chunks):
                print(f"\n[청크 {chunk_idx + 1}]")
                if hasattr(chunk, "page_content"):
                    print(chunk.page_content)
                else:
                    print("청크 내용을 가져올 수 없습니다.")
                print("-" * 50)
else:
    print("로드된 문서가 없습니다.")


In [ ]:
from typing import Dict, List, Union

import tiktoken
from langchain_core.documents import Document

# tiktoken 인코더 초기화
encoder: tiktoken.Encoding = tiktoken.get_encoding("o200k_base")


def count_tokens(text: str) -> int:
    return len(encoder.encode(text))


def check_chunk_tokens(chunk: Document) -> int:
    return count_tokens(chunk.page_content)


def calculate_document_tokens(doc_chunks: List[Document]) -> int:
    total_tokens: int = 0
    for chunk in doc_chunks:
        total_tokens += count_tokens(chunk.page_content)
    return total_tokens


def analyze_document_tokens(doc_chunks: List[Document]) -> Dict[str, Union[int, float]]:
    if not doc_chunks:
        return {
            "total_chunks": 0,
            "total_tokens": 0,
            "avg_tokens": 0.0,
            "max_tokens": 0,
            "min_tokens": 0,
        }

    token_counts: List[int] = [check_chunk_tokens(chunk) for chunk in doc_chunks]

    return {
        "total_chunks": len(doc_chunks),
        "total_tokens": sum(token_counts),
        "avg_tokens": sum(token_counts) / len(token_counts),
        "max_tokens": max(token_counts),
        "min_tokens": min(token_counts),
    }


def analyze_file_tokens(file_chunks: List[List[Document]]) -> Dict[str, Union[int, float]]:
    all_chunks: List[Document] = []
    for doc_chunks in file_chunks:
        all_chunks.extend(doc_chunks)

    return analyze_document_tokens(all_chunks)


In [ ]:
# Document별 청크당 평균 토큰 수 테스트
if "chunked_result" in locals() and chunked_result:
    for file_idx, file_chunks in enumerate(chunked_result):
        print(f"파일 {file_idx + 1}:")
        for doc_idx, doc_chunks in enumerate(file_chunks):
            doc_stats: Dict[str, Union[int, float]] = analyze_document_tokens(doc_chunks)
            avg_tokens_per_chunk: float = float(doc_stats["avg_tokens"])
            print(
                f"  Document {doc_idx + 1}: 청크 {doc_stats['total_chunks']}개, 청크당 평균 {avg_tokens_per_chunk:.2f} 토큰"
            )
else:
    print("청킹 결과가 없습니다.")


In [ ]:
from typing import Dict, List, Union, cast

from langchain_core.documents import Document


def create_sequential_batches(
    doc_chunks: List[Document], max_tokens: int = 8191
) -> List[List[Document]]:
    """순차적 방식으로 배치 생성"""
    if not doc_chunks:
        return []

    # 작은 문서는 단일 배치로 처리
    total_tokens: int = sum(count_tokens(chunk.page_content) for chunk in doc_chunks)
    if total_tokens <= max_tokens:
        return [doc_chunks]

    # 순차적 방식으로 배치 생성
    batches: List[List[Document]] = []
    current_batch: List[Document] = []
    current_tokens: int = 0

    for chunk in doc_chunks:
        chunk_tokens: int = count_tokens(chunk.page_content)
        if current_tokens + chunk_tokens <= max_tokens:
            current_batch.append(chunk)
            current_tokens += chunk_tokens
        else:
            if current_batch:
                batches.append(current_batch)
            current_batch = [chunk]
            current_tokens = chunk_tokens

    if current_batch:
        batches.append(current_batch)
    return batches


def process_document_batches(
    doc_chunks: List[Document],
    file_idx: int,
    doc_idx: int,
    max_tokens: int = 8191,
) -> List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]]:
    batches: List[List[Document]] = create_sequential_batches(doc_chunks, max_tokens)
    processed: List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]] = []

    # 청크 인덱스 매핑 및 메타데이터 추가
    running_idx: int = 0
    for batch_idx, batch in enumerate(batches):
        batch_tokens: int = sum(count_tokens(c.page_content) for c in batch)
        chunk_indices: List[int] = list(range(running_idx, running_idx + len(batch)))
        running_idx += len(batch)

        processed.append(
            {
                "batch": batch,
                "metadata": {
                    "file_idx": file_idx,
                    "doc_idx": doc_idx,
                    "batch_idx": batch_idx,
                    "chunk_count": len(batch),
                    "chunk_indices": chunk_indices,
                    "total_tokens": batch_tokens,
                    "utilization_rate": batch_tokens / max_tokens,
                },
            }
        )
    return processed


def process_all_documents(
    chunked_by_file: List[List[List[Document]]],
    max_tokens: int = 8191,
) -> List[List[List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]]]]:
    all_processed: List[
        List[List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]]]
    ] = []

    # 모든 파일과 문서에 대해 배치 처리
    for file_idx, file_chunks in enumerate(chunked_by_file):
        file_processed: List[
            List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]]
        ] = []

        for doc_idx, doc_chunks in enumerate(file_chunks):
            file_processed.append(
                process_document_batches(doc_chunks, file_idx, doc_idx, max_tokens)
            )
        all_processed.append(file_processed)
    return all_processed


def analyze_batch_statistics(
    processed_batches: List[
        List[List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]]]
    ],
) -> Dict[str, Union[int, float]]:
    total_batches: int = 0
    total_tokens: int = 0
    utilization_rates: List[float] = []

    # 모든 배치의 통계 수집
    for file_batches in processed_batches:
        for doc_batches in file_batches:
            for batch_info in doc_batches:
                total_batches += 1
                metadata = cast(Dict[str, Union[int, float, List[int]]], batch_info["metadata"])
                total_tokens += int(cast(int, metadata["total_tokens"]))
                utilization_rates.append(float(cast(float, metadata["utilization_rate"])))

    # 활용률 통계 계산
    avg_utilization: float = (
        (sum(utilization_rates) / len(utilization_rates)) if utilization_rates else 0.0
    )
    min_utilization: float = min(utilization_rates) if utilization_rates else 0.0
    max_utilization: float = max(utilization_rates) if utilization_rates else 0.0

    return {
        "total_batches": total_batches,
        "total_tokens": total_tokens,
        "avg_utilization_rate": avg_utilization,
        "min_utilization_rate": min_utilization,
        "max_utilization_rate": max_utilization,
    }


In [ ]:
# 순차적 배치 테스트
from statistics import mean
from typing import Dict, List, Union, cast

assert "chunked_result" in locals() and chunked_result, "chunked_result가 없습니다."

file_result: List[List[Document]] = chunked_result[0]
doc_idx: int = 0
doc_chunks: List[Document] = file_result[doc_idx]

processed_batches: List[
    Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]
] = process_document_batches(doc_chunks, file_idx=test_file_idx, doc_idx=doc_idx, max_tokens=8191)

# 배치 통계 수집
util_rates: List[float] = []
max_tokens_each: List[int] = []
for batch in processed_batches:
    stats_meta: Dict[str, Union[int, float, List[int]]] = cast(
        Dict[str, Union[int, float, List[int]]], batch["metadata"]
    )
    util_rates.append(float(cast(float, stats_meta["utilization_rate"])))
    max_tokens_each.append(int(cast(int, stats_meta["total_tokens"])))

print(f"파일 idx: {test_file_idx}, Document idx: {doc_idx}")
print(f"배치 수: {len(processed_batches)}")
print(f"배치별 총 토큰(상위 5개): {sorted(max_tokens_each, reverse=True)[:5]}")
print(
    f"활용률 평균: {mean(util_rates):.2%}, 최소: {min(util_rates):.2%}, 최대: {max(util_rates):.2%}"
)

# 제약 검증
violations: List[int] = [t for t in max_tokens_each if t > 8191]
print(f"8191 초과 배치 수: {len(violations)}")

# 배치 메타데이터 출력
for batch in processed_batches:
    print_meta: Dict[str, Union[int, float, List[int]]] = cast(
        Dict[str, Union[int, float, List[int]]], batch["metadata"]
    )
    indices_str: str = str(cast(List[int], print_meta["chunk_indices"])[:10]) + (
        " ..." if len(cast(List[int], print_meta["chunk_indices"])) > 10 else ""
    )
    print(
        f"batch_idx={cast(int, print_meta['batch_idx'])}, chunk_count={cast(int, print_meta['chunk_count'])}, total_tokens={cast(int, print_meta['total_tokens'])}, utilization={cast(float, print_meta['utilization_rate']):.2%}, indices={indices_str}"
    )


In [ ]:
# 이진탐색 기반 정확한 배치 생성 (배치 크기 초과 방지)
from typing import Dict, List, Tuple, Union, cast

from langchain_core.documents import Document


def split_text_by_size(text: str, chunk_size: int, ext: str) -> List[str]:
    # 확장자별 설정 가져오기
    ext_config: Dict[str, Union[int, bool]] = CHUNKING_CONFIG.get(ext, {})
    rechunk_overlap: int = ext_config.get("chunk_overlap", DEFAULT_CHUNK_OVERLAP) // 2

    if rechunk_overlap >= chunk_size:
        rechunk_overlap = max(chunk_size // 2, 1)

    # 마크다운 분할기 또는 일반 분할기 선택
    if ext_config.get("use_markdown_splitter", False):
        splitter = MarkdownTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=rechunk_overlap,
        )
    else:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=rechunk_overlap,
            length_function=len,
        )

    return splitter.split_text(text)


def rechunk_for_batch_optimization(
    chunk: Document, remaining_tokens: int, ext: str
) -> Tuple[List[Document], List[Document]]:
    chunk_text: str = chunk.page_content
    current_tokens: int = count_tokens(chunk_text)

    if current_tokens <= remaining_tokens:
        return [chunk], []

    # 이진탐색으로 정확한 분할점 찾기
    left: int = 1
    right: int = len(chunk_text)
    best_size: int = 1

    while left <= right:
        mid: int = (left + right) // 2
        test_text: str = chunk_text[:mid]
        test_chunks: List[str] = split_text_by_size(test_text, mid, ext)

        if test_chunks:
            # 모든 청크의 총 토큰 수 확인
            total_test_tokens: int = sum(count_tokens(text) for text in test_chunks)
            if total_test_tokens <= remaining_tokens:
                best_size = mid
                left = mid + 1
            else:
                right = mid - 1
        else:
            right = mid - 1

    # front/back 청크 생성
    front_chunks: List[Document] = []
    back_chunks: List[Document] = []

    if best_size > 0:
        front_text: str = chunk_text[:best_size]
        front_texts: List[str] = split_text_by_size(front_text, best_size, ext)
        # 메타데이터 타입 안전하게 처리
        front_metadata: Dict[str, str] = cast(Dict[str, str], chunk.metadata)
        front_chunks = [
            Document(page_content=text, metadata=dict(front_metadata)) for text in front_texts
        ]

    remaining_text: str = chunk_text[best_size:]
    if remaining_text.strip():
        # 메타데이터 타입 안전하게 처리
        back_metadata: Dict[str, str] = cast(Dict[str, str], chunk.metadata)
        back_chunks = [Document(page_content=remaining_text, metadata=dict(back_metadata))]

    return front_chunks, back_chunks


def create_precise_batches(
    doc_chunks: List[Document], ext: str, max_tokens: int = 8191
) -> List[List[Document]]:
    if not doc_chunks:
        return []

    batches: List[List[Document]] = []
    current_batch: List[Document] = []
    current_tokens: int = 0

    for chunk in doc_chunks:
        chunk_tokens: int = count_tokens(chunk.page_content)

        if current_tokens + chunk_tokens <= max_tokens:
            current_batch.append(chunk)
            current_tokens += chunk_tokens
        else:
            if current_batch:
                remaining_tokens: int = max_tokens - current_tokens
                if remaining_tokens > 0:
                    # 청크를 분할하여 배치 최적화
                    front_chunks, back_chunks = rechunk_for_batch_optimization(
                        chunk, remaining_tokens, ext
                    )
                    current_batch.extend(front_chunks)
                    batches.append(current_batch)
                    current_batch = back_chunks
                    current_tokens = sum(count_tokens(c.page_content) for c in current_batch)
                else:
                    batches.append(current_batch)
                    current_batch = [chunk]
                    current_tokens = chunk_tokens
            else:
                current_batch = [chunk]
                current_tokens = chunk_tokens

    if current_batch:
        batches.append(current_batch)

    return batches


def process_document_batches_precise(
    doc_chunks: List[Document],
    file_idx: int,
    doc_idx: int,
    ext: str,
    max_tokens: int = 8191,
) -> List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]]:
    batches: List[List[Document]] = create_precise_batches(doc_chunks, ext, max_tokens)
    processed: List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]] = []

    running_idx: int = 0
    for batch_idx, batch in enumerate(batches):
        batch_tokens: int = sum(count_tokens(c.page_content) for c in batch)
        chunk_indices: List[int] = list(range(running_idx, running_idx + len(batch)))
        running_idx += len(batch)

        processed.append(
            {
                "batch": batch,
                "metadata": {
                    "file_idx": file_idx,
                    "doc_idx": doc_idx,
                    "batch_idx": batch_idx,
                    "chunk_count": len(batch),
                    "chunk_indices": chunk_indices,
                    "total_tokens": batch_tokens,
                    "utilization_rate": batch_tokens / max_tokens,
                },
            }
        )

    return processed


In [ ]:
# 이진탐색 배치 테스트
from statistics import mean
from typing import Dict, List, Union, cast

from langchain_core.documents import Document

assert "chunked_result" in locals() and chunked_result, "chunked_result가 없습니다."

file_result: List[List[Document]] = chunked_result[0]
doc_idx: int = 0
doc_chunks: List[Document] = file_result[doc_idx]

processed_batches_precise: List[
    Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]
] = process_document_batches_precise(
    doc_chunks, file_idx=test_file_idx, doc_idx=doc_idx, ext=".txt", max_tokens=8191
)

# 배치 통계 수집
util_rates_p: List[float] = []
max_tokens_each_p: List[int] = []
for batch in processed_batches_precise:
    stats_meta: Dict[str, Union[int, float, List[int]]] = cast(
        Dict[str, Union[int, float, List[int]]], batch["metadata"]
    )
    util_rates_p.append(float(cast(float, stats_meta["utilization_rate"])))
    max_tokens_each_p.append(int(cast(int, stats_meta["total_tokens"])))

print(f"파일 idx: {test_file_idx}, Document idx: {doc_idx}")
print(f"배치 수: {len(processed_batches_precise)}")
print(f"배치별 총 토큰(상위 5개): {sorted(max_tokens_each_p, reverse=True)[:5]}")
print(
    f"활용률 평균: {mean(util_rates_p):.2%}, 최소: {min(util_rates_p):.2%}, 최대: {max(util_rates_p):.2%}"
)

# 제약 검증
violations_p: List[int] = [t for t in max_tokens_each_p if t > 8191]
print(f"8191 초과 배치 수: {len(violations_p)}")

# 배치 메타데이터 출력
for batch in processed_batches_precise:
    print_meta: Dict[str, Union[int, float, List[int]]] = cast(
        Dict[str, Union[int, float, List[int]]], batch["metadata"]
    )
    indices_str: str = str(cast(List[int], print_meta["chunk_indices"])[:10]) + (
        " ..." if len(cast(List[int], print_meta["chunk_indices"])) > 10 else ""
    )
    print(
        f"batch_idx={cast(int, print_meta['batch_idx'])}, chunk_count={cast(int, print_meta['chunk_count'])}, total_tokens={cast(int, print_meta['total_tokens'])}, utilization={cast(float, print_meta['utilization_rate']):.2%}, indices={indices_str}"
    )


In [ ]:
# 토큰 기반 배치 생성 함수
from typing import Dict, List, Tuple, Union, cast

from langchain_core.documents import Document
from langchain_text_splitters import MarkdownTextSplitter, RecursiveCharacterTextSplitter


def create_precise_batches(
    doc_chunks: List[Document], ext: str, max_tokens: int = 8191
) -> List[List[Document]]:
    """정확한 배치 경계를 찾아 토큰 단위로 분할하는 배치 생성"""
    if not doc_chunks:
        return []

    batches: List[List[Document]] = []
    current_batch: List[Document] = []
    current_tokens: int = 0

    for _, chunk in enumerate(doc_chunks):
        chunk_tokens: int = count_tokens(chunk.page_content)

        if current_tokens + chunk_tokens <= max_tokens:
            current_batch.append(chunk)
            current_tokens += chunk_tokens
        else:
            if current_batch:
                remaining_tokens: int = max_tokens - current_tokens

                if remaining_tokens > 0:
                    front_chunks, back_chunks = rechunk_for_batch_optimization(
                        chunk, remaining_tokens, ext
                    )

                    current_batch.extend(front_chunks)
                    batches.append(current_batch)

                    if back_chunks:
                        current_batch = back_chunks
                        current_tokens = sum(count_tokens(c.page_content) for c in back_chunks)
                    else:
                        current_batch = []
                        current_tokens = 0
                else:
                    batches.append(current_batch)
                    current_batch = [chunk]
                    current_tokens = chunk_tokens
            else:
                current_batch = [chunk]
                current_tokens = chunk_tokens

    if current_batch:
        batches.append(current_batch)

    return batches


def rechunk_for_batch_optimization(
    chunk: Document, remaining_tokens: int, ext: str
) -> Tuple[List[Document], List[Document]]:
    """정확한 토큰 수로 분할"""
    chunk_text: str = chunk.page_content
    current_tokens: int = count_tokens(chunk_text)

    if current_tokens <= remaining_tokens:
        return [chunk], []

    tokens = encoder.encode(chunk_text)

    if len(tokens) <= remaining_tokens:
        return [chunk], []

    front_tokens = tokens[:remaining_tokens]
    back_tokens = tokens[remaining_tokens:]

    front_text = encoder.decode(front_tokens)
    back_text = encoder.decode(back_tokens)

    # metadata 타입 명시적 캐스팅
    chunk_metadata: Dict[str, str] = cast(Dict[str, str], chunk.metadata)

    front_chunks = [Document(page_content=front_text, metadata=chunk_metadata)]
    back_chunks = (
        [Document(page_content=back_text, metadata=chunk_metadata)] if back_text.strip() else []
    )

    return front_chunks, back_chunks


def split_text_by_size(text: str, chunk_size: int, ext: str) -> List[str]:
    """지정된 크기로 텍스트 분할"""
    ext_config: Dict[str, Union[int, bool]] = CHUNKING_CONFIG.get(ext, {})
    rechunk_overlap: int = ext_config.get("chunk_overlap", DEFAULT_CHUNK_OVERLAP) // 2

    if ext_config.get("use_markdown_splitter", False):
        splitter = MarkdownTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=rechunk_overlap,
        )
    else:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=rechunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""],
        )

    return splitter.split_text(text)


def process_document_batches_precise(
    doc_chunks: List[Document],
    file_idx: int,
    doc_idx: int,
    ext: str,
    max_tokens: int = 8191,
) -> List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]]:
    """정확한 배치 경계 기반으로 Document별 배치 처리"""
    batches = create_precise_batches(doc_chunks, ext, max_tokens)
    processed: List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]] = []

    running_idx = 0
    for batch_idx, batch in enumerate(batches):
        batch_tokens = sum(count_tokens(c.page_content) for c in batch)
        chunk_indices = list(range(running_idx, running_idx + len(batch)))
        running_idx += len(batch)

        processed.append(
            {
                "batch": batch,
                "metadata": {
                    "file_idx": file_idx,
                    "doc_idx": doc_idx,
                    "batch_idx": batch_idx,
                    "chunk_count": len(batch),
                    "chunk_indices": chunk_indices,
                    "total_tokens": batch_tokens,
                    "utilization_rate": batch_tokens / max_tokens,
                },
            }
        )

    return processed


In [ ]:
# 토큰 기반 분할 배치 테스트
from statistics import mean
from typing import Dict, List, Union, cast

from langchain_core.documents import Document

assert "chunked_result" in locals() and chunked_result, "chunked_result가 없습니다."

file_result: List[List[Document]] = chunked_result[0]
doc_idx: int = 0
doc_chunks: List[Document] = file_result[doc_idx]

processed_batches_token: List[
    Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]
] = process_document_batches_precise(
    doc_chunks, file_idx=test_file_idx, doc_idx=doc_idx, ext=".txt", max_tokens=8191
)

# 배치 통계 수집
util_rates_t: List[float] = []
max_tokens_each_t: List[int] = []
for batch in processed_batches_token:
    stats_meta: Dict[str, Union[int, float, List[int]]] = cast(
        Dict[str, Union[int, float, List[int]]], batch["metadata"]
    )
    util_rates_t.append(float(cast(float, stats_meta["utilization_rate"])))
    max_tokens_each_t.append(int(cast(int, stats_meta["total_tokens"])))

print(f"파일 idx: {test_file_idx}, Document idx: {doc_idx}")
print(f"배치 수: {len(processed_batches_token)}")
print(f"배치별 총 토큰(상위 5개): {sorted(max_tokens_each_t, reverse=True)[:5]}")
print(
    f"활용률 평균: {mean(util_rates_t):.2%}, 최소: {min(util_rates_t):.2%}, 최대: {max(util_rates_t):.2%}"
)

# 제약 검증
violations_t: List[int] = [t for t in max_tokens_each_t if t > 8191]
print(f"8191 초과 배치 수: {len(violations_t)}")

# 배치 메타데이터 출력
for batch in processed_batches_token:
    print_meta: Dict[str, Union[int, float, List[int]]] = cast(
        Dict[str, Union[int, float, List[int]]], batch["metadata"]
    )
    indices_str: str = str(cast(List[int], print_meta["chunk_indices"])[:10]) + (
        " ..." if len(cast(List[int], print_meta["chunk_indices"])) > 10 else ""
    )
    print(
        f"batch_idx={cast(int, print_meta['batch_idx'])}, chunk_count={cast(int, print_meta['chunk_count'])}, total_tokens={cast(int, print_meta['total_tokens'])}, utilization={cast(float, print_meta['utilization_rate']):.2%}, indices={indices_str}"
    )


In [ ]:
# 휴리스틱 배치 생성 함수
from typing import Dict, List, Union, cast

from langchain_core.documents import Document
from langchain_text_splitters import MarkdownTextSplitter, RecursiveCharacterTextSplitter


def create_heuristic_batches(
    doc_chunks: List[Document], ext: str, max_tokens: int = 8191
) -> List[List[Document]]:
    """평균값 기반 휴리스틱으로 배치 생성"""
    if not doc_chunks:
        return []

    total_tokens: int = sum(count_tokens(chunk.page_content) for chunk in doc_chunks)
    avg_tokens: float = total_tokens / len(doc_chunks)
    chunks_per_batch: int = max_tokens // int(avg_tokens)

    rechunked_chunks: List[Document] = []
    rechunk_count: int = 0

    for i, chunk in enumerate(doc_chunks):
        if (i + 1) % chunks_per_batch == 0:
            splitter: Union[MarkdownTextSplitter, RecursiveCharacterTextSplitter] = get_splitter(
                ext
            )

            if isinstance(splitter, MarkdownTextSplitter):
                markdown_texts: List[str] = splitter.split_text(chunk.page_content)
                chunk_metadata: Dict[str, str] = cast(Dict[str, str], chunk.metadata)
                rechunked_chunks.extend(
                    [
                        Document(page_content=text, metadata=chunk_metadata)
                        for text in markdown_texts
                    ]
                )
            else:
                recursive_chunks: List[Document] = splitter.split_documents([chunk])
                rechunked_chunks.extend(recursive_chunks)

            rechunk_count += 1
        else:
            rechunked_chunks.append(chunk)

    batches: List[List[Document]] = []
    current_batch: List[Document] = []
    current_tokens: int = 0

    for chunk in rechunked_chunks:
        chunk_tokens: int = count_tokens(chunk.page_content)

        if current_tokens + chunk_tokens <= max_tokens:
            current_batch.append(chunk)
            current_tokens += chunk_tokens
        else:
            if current_batch:
                batches.append(current_batch)
            current_batch = [chunk]
            current_tokens = chunk_tokens

    if current_batch:
        batches.append(current_batch)

    return batches


def process_document_batches_heuristic(
    doc_chunks: List[Document],
    file_idx: int,
    doc_idx: int,
    ext: str,
    max_tokens: int = 8191,
) -> List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]]:
    """평균값 기반 휴리스틱으로 Document별 배치 처리"""
    batches: List[List[Document]] = create_heuristic_batches(doc_chunks, ext, max_tokens)
    processed: List[Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]] = []

    running_idx: int = 0
    for batch_idx, batch in enumerate(batches):
        batch_tokens: int = sum(count_tokens(c.page_content) for c in batch)
        chunk_indices: List[int] = list(range(running_idx, running_idx + len(batch)))
        running_idx += len(batch)

        processed.append(
            {
                "batch": batch,
                "metadata": {
                    "file_idx": file_idx,
                    "doc_idx": doc_idx,
                    "batch_idx": batch_idx,
                    "chunk_count": len(batch),
                    "chunk_indices": chunk_indices,
                    "total_tokens": batch_tokens,
                    "utilization_rate": batch_tokens / max_tokens,
                },
            }
        )

    return processed


In [ ]:
# 휴리스틱 배치 테스트
from statistics import mean
from typing import Dict, List, Union, cast

from langchain_core.documents import Document

assert "chunked_result" in locals() and chunked_result, "chunked_result가 없습니다."

file_result: List[List[Document]] = chunked_result[0]
doc_idx: int = 0
doc_chunks: List[Document] = file_result[doc_idx]

processed_batches_heuristic: List[
    Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]
] = process_document_batches_heuristic(
    doc_chunks, file_idx=test_file_idx, doc_idx=doc_idx, ext=".txt", max_tokens=8191
)

# 배치 통계 수집
util_rates_h: List[float] = []
max_tokens_each_h: List[int] = []
for batch in processed_batches_heuristic:
    stats_meta: Dict[str, Union[int, float, List[int]]] = cast(
        Dict[str, Union[int, float, List[int]]], batch["metadata"]
    )
    util_rates_h.append(float(cast(float, stats_meta["utilization_rate"])))
    max_tokens_each_h.append(int(cast(int, stats_meta["total_tokens"])))

print(f"파일 idx: {test_file_idx}, Document idx: {doc_idx}")
print(f"배치 수: {len(processed_batches_heuristic)}")
print(f"배치별 총 토큰(상위 5개): {sorted(max_tokens_each_h, reverse=True)[:5]}")
print(
    f"활용률 평균: {mean(util_rates_h):.2%}, 최소: {min(util_rates_h):.2%}, 최대: {max(util_rates_h):.2%}"
)

# 제약 검증
violations_h: List[int] = [t for t in max_tokens_each_h if t > 8191]
print(f"8191 초과 배치 수: {len(violations_h)}")

# 배치 메타데이터 출력
for batch in processed_batches_heuristic:
    print_meta: Dict[str, Union[int, float, List[int]]] = cast(
        Dict[str, Union[int, float, List[int]]], batch["metadata"]
    )
    indices_str: str = str(cast(List[int], print_meta["chunk_indices"])[:10]) + (
        " ..." if len(cast(List[int], print_meta["chunk_indices"])) > 10 else ""
    )
    print(
        f"batch_idx={cast(int, print_meta['batch_idx'])}, chunk_count={cast(int, print_meta['chunk_count'])}, total_tokens={cast(int, print_meta['total_tokens'])}, utilization={cast(float, print_meta['utilization_rate']):.2%}, indices={indices_str}"
    )


In [ ]:
# 실제 파일 기반 스플리터 성능 비교 테스트
import time
from typing import Dict, List

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter


def performance_test_real_files(docs: List[List[Document]]) -> Dict[str, Dict[str, float]]:
    """실제 파일 기반 스플리터 성능 비교 테스트"""
    print("=== 실제 파일 기반 스플리터 성능 비교 테스트 ===")

    if not docs:
        print("docs가 비어있습니다.")
        return {}

    test_texts: Dict[str, str] = {}
    for i, file_docs in enumerate(docs[:5]):  # 처음 5개 파일만 테스트
        if file_docs:
            test_texts[f"file_{i}"] = file_docs[0].page_content[:50000]  # 50,000자로 제한

    if not test_texts:
        print("테스트할 파일이 없습니다.")
        return {}

    iterations: int = 50
    results: Dict[str, Dict[str, float]] = {}

    for text_name, test_text in test_texts.items():
        print(f"\n--- {text_name} 테스트 ---")
        print(f"텍스트 길이: {len(test_text):,}자")

        char_splitter: CharacterTextSplitter = CharacterTextSplitter(
            chunk_size=1000, chunk_overlap=100, separator=" "
        )
        recursive_splitter: RecursiveCharacterTextSplitter = RecursiveCharacterTextSplitter(
            chunk_size=1000, chunk_overlap=100
        )

        # CharacterTextSplitter 테스트
        char_chunks: List[str] = []
        start_time: float = time.time()
        for _ in range(iterations):
            char_chunks = char_splitter.split_text(test_text)
        char_time: float = time.time() - start_time

        # RecursiveCharacterTextSplitter 테스트
        recursive_chunks: List[str] = []
        start_time = time.time()
        for _ in range(iterations):
            recursive_chunks = recursive_splitter.split_text(test_text)
        recursive_time: float = time.time() - start_time

        # 결과 계산
        char_avg_time: float = char_time / iterations
        recursive_avg_time: float = recursive_time / iterations
        time_diff: float = recursive_avg_time - char_avg_time
        overhead_percent: float = (time_diff / char_avg_time) * 100 if char_avg_time > 0 else 0

        # 결과 출력
        print(f"CharacterTextSplitter: {char_avg_time:.6f}초/회, {len(char_chunks)}개 청크")
        print(
            f"RecursiveCharacterTextSplitter: {recursive_avg_time:.6f}초/회, {len(recursive_chunks)}개 청크"
        )
        print(f"시간 차이: {time_diff:.6f}초 ({overhead_percent:+.2f}%)")

        # 청크 크기 분석
        char_avg_chunk_size: float = (
            sum(len(chunk) for chunk in char_chunks) / len(char_chunks) if char_chunks else 0
        )
        recursive_avg_chunk_size: float = (
            sum(len(chunk) for chunk in recursive_chunks) / len(recursive_chunks)
            if recursive_chunks
            else 0
        )

        print(
            f"평균 청크 크기 - Character: {char_avg_chunk_size:.1f}자, Recursive: {recursive_avg_chunk_size:.1f}자"
        )

        results[text_name] = {
            "char_time": char_avg_time,
            "recursive_time": recursive_avg_time,
            "char_chunks": float(len(char_chunks)),
            "recursive_chunks": float(len(recursive_chunks)),
            "overhead_percent": overhead_percent,
            "char_avg_chunk_size": char_avg_chunk_size,
            "recursive_avg_chunk_size": recursive_avg_chunk_size,
        }

    # 전체 요약
    print(f"\n=== 전체 요약 ===")
    total_char_time: float = sum(r["char_time"] for r in results.values())
    total_recursive_time: float = sum(r["recursive_time"] for r in results.values())
    total_overhead: float = ((total_recursive_time - total_char_time) / total_char_time) * 100

    print(
        f"전체 평균 - Character: {total_char_time:.6f}초, Recursive: {total_recursive_time:.6f}초"
    )
    print(f"전체 오버헤드: {total_overhead:+.2f}%")

    return results


# 성능 테스트 실행
performance_results: Dict[str, Dict[str, float]] = performance_test_real_files(docs)


## 적합한 청커 찾기
